# Fine-tuning IndoBERT untuk Sentimen BSI
Notebook ini dijalankan di Google Colab dengan GPU.

**Sebelum mulai:** Runtime > Change runtime type > pilih **T4 GPU**.

Langkah:
1. Install dependencies
2. Upload database SQLite Anda
3. Jalankan pipeline: load → clean → split → fine-tune → evaluasi jujur di test set


In [ ]:
!pip install -q transformers datasets scikit-learn accelerate seaborn matplotlib

## 1. Upload database SQLite Anda
Jalankan cell di bawah, lalu pilih file `.db` / `.sqlite` Anda.

In [ ]:
from google.colab import files
uploaded = files.upload()
DB_FILENAME = list(uploaded.keys())[0]
print(f"File terupload: {DB_FILENAME}")

## 2. Cek dulu isi tabel `datasets` (sanity check sebelum training)

In [ ]:
import sqlite3, json, pandas as pd

conn = sqlite3.connect(DB_FILENAME)
meta = pd.read_sql_query("SELECT nama_dataset, deskripsi FROM datasets", conn)
conn.close()
print(meta)

## 3. Jalankan pipeline fine-tuning lengkap
Upload juga file `finetune_indobert.py` ke Colab (folder kiri > upload), lalu sesuaikan `DB_PATH` di dalamnya ke nama file yang baru diupload, atau jalankan langsung lewat cell ini.

In [ ]:
# Edit DB_PATH di file finetune_indobert.py agar sesuai nama file upload Anda,
# atau jalankan baris berikut untuk override otomatis sebelum running:

import re
with open("finetune_indobert.py") as f:
    content = f.read()
content = re.sub(r'DB_PATH = ".*"', f'DB_PATH = "{DB_FILENAME}"', content)
with open("finetune_indobert.py", "w") as f:
    f.write(content)

print("DB_PATH sudah disesuaikan. Menjalankan training...")
!python finetune_indobert.py

## 4. Lihat hasil evaluasi
Setelah training selesai, cek file-file berikut yang otomatis tersimpan:
- `indobert_bsi_sentiment_classification_report.txt` -- precision/recall/f1 per kelas (untuk dilampirkan di skripsi)
- `indobert_bsi_sentiment_confusion_matrix.png` -- visualisasi confusion matrix
- `indobert_bsi_sentiment_test_predictions.csv` -- detail prediksi per tweet di test set, untuk audit manual

**Penting:** angka di classification report inilah akurasi yang JUJUR untuk dilaporkan di skripsi, karena test set sama sekali tidak pernah dipakai untuk training atau menulis aturan apa pun.

In [ ]:
print(open("indobert_bsi_sentiment_classification_report.txt").read())

In [ ]:
from IPython.display import Image
Image("indobert_bsi_sentiment_confusion_matrix.png")

## 5. Download model hasil fine-tuning (opsional, untuk dipakai lagi nanti)

In [ ]:
!zip -r indobert_bsi_sentiment.zip indobert_bsi_sentiment
from google.colab import files
files.download("indobert_bsi_sentiment.zip")

## 6. Pakai model untuk dataset baru (tanpa training ulang)
Upload `predict_new_data.py`, lalu jalankan untuk dataset lain di tabel yang sama tanpa perlu menulis regex apa pun lagi.

In [ ]:
# Ganti "nama_dataset_baru" dengan nilai kolom nama_dataset yang ingin diuji
import re
with open("predict_new_data.py") as f:
    content = f.read()
content = re.sub(r'DB_PATH = ".*"', f'DB_PATH = "{DB_FILENAME}"', content)
with open("predict_new_data.py", "w") as f:
    f.write(content)

!python predict_new_data.py --dataset_name "nama_dataset_baru" 